# Chapter 7: Uncertainty Estimation

This notebook accompanies **Chapter 7** of the lecture notes.

> Last lecture's VAE could reconstruct any input and dream up new ones from the prior. Neither capability told us how much to trust either output. Today the model learns the second skill: knowing what it doesn't know.

**Agenda**

🪤 · 🎲 · 🪞 · 🎁 · ⚡ · 🏁

**Take it from here:** 🧭 · 🚦 · 🛡️ · 🌡️

> **Tip:** Run cells top to bottom. Later cells depend on earlier ones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys; sys.path.insert(0, '../..')
from plot_style import *
from checks import (
    check_predictive_stats,
)


## 🪤 The Confidence Trap

A digit classifier was trained for you, but with a twist: it only ever saw the **even** digits 0, 2, 4, 6, 8. The **odd** digits 1, 3, 5, 7, 9 are entirely outside its training distribution; the model has no class for them, and no idea they exist.

This setup is the microscope of the chapter. Whenever we want to ask "what should the model say about an input it has never seen?", we hand it an odd digit and watch what comes out.

> The classifier returns a class label and a number called confidence. The label will, by construction, be wrong on every odd digit. Will the confidence number tell you so?

<details><summary>Thought</summary>

Almost certainly not. The softmax normalises across the five known classes; whichever even class the input most resembles wins, and the resulting probability has no way to communicate "and by the way, this is none of the above". A 1 that looks faintly like a 7 will be confidently called a 7. A confident answer is not the same as a correct one.
</details>

The cells below load the classifier's predictions on a held-out test set (about 1000 evens and 1000 odds, mixed) and ask the same question with pictures.


In [ ]:
images = pd.read_csv('test_images.csv')
preds  = pd.read_csv('clf_predictions.csv')
df     = preds.merge(images.drop(columns=['label', 'parity']), on='idx')

print(f'Total test samples: {len(df)}')
print(f"  evens (in-distribution): {(df['parity'] == 'even').sum()}")
print(f"  odds  (out-of-dist)    : {(df['parity'] == 'odd').sum()}")


In [ ]:
# Ten odd-digit inputs the classifier was *most* confident about.
# These indices are hand-picked after the fact to land a clean visual; if you
# regenerate the assets they may shift. The 'sort by confidence' fallback below
# always works.
_TRAP_INDICES = [79, 211, 219, 286, 864, 972, 1296, 1472, 1499, 1818]

if _TRAP_INDICES is None:
    _odd = df[df['parity'] == 'odd'].sort_values('top1_softmax', ascending=False).head(10)
else:
    _odd = df[df['idx'].isin(_TRAP_INDICES)].sort_values('idx')

_pix_cols = [f'pixel_{j}' for j in range(784)]
fig, axes = plt.subplots(2, 10, figsize=(14, 3.6))
for col, (_, row) in enumerate(_odd.iterrows()):
    img = row[_pix_cols].values.astype(float).reshape(28, 28)
    axes[0, col].imshow(img, cmap='magma')
    axes[0, col].set_title(f"true: {int(row['label'])}", fontsize=9, color=_GOLDEN)
    axes[1, col].imshow(img, cmap='magma', alpha=0.25)
    axes[1, col].text(14, 14, str(int(row['top1_label'])), ha='center', va='center',
                      fontsize=22, color=_ACCENT, fontweight='bold')
    axes[1, col].set_title(f"p = {row['top1_softmax']:.2f}", fontsize=9, color=_ACCENT)
    for ax in (axes[0, col], axes[1, col]):
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_visible(False)
axes[0, 0].set_ylabel('input', color=_TEXT)
axes[1, 0].set_ylabel('model says', color=_TEXT)
plt.tight_layout()
plt.show()


**Observe:**
- Every digit in the top row is odd. The model has never been trained to recognise any of them.
- The bottom row shows what the classifier said anyway, and how confident it was. Numbers above 0.9 are common; the model declares "this is a 4" with the same vigour it would on a real 4.
- The softmax has no opinion on whether the input belongs to one of its known classes at all. It normalises across them, picks the largest, reports the result.


### Two flavours of uncertainty

The failures above split into two categories that are worth keeping separate. The distinction decides what to do about each.

**Aleatoric uncertainty** is irreducible noise in the data. Two classes that genuinely overlap, a sensor with finite precision, a fundamentally ambiguous input. No amount of training data lifts the floor; the optimal model still returns a probability, never a label. A 9 that any reader would also call a 4 belongs here.

**Epistemic uncertainty** reflects what the model has not seen. It shrinks the moment relevant data arrives. A model trained on summer temperatures has high epistemic uncertainty about winter; show it winter data and the gap closes. The 1 confidently labelled a 2 is the textbook case: the model has no concept of "1" at all, and the doubt should have been visible in the answer.

Aleatoric uncertainty is for honest reporting. Epistemic uncertainty is for action: collect more data, route to a human reviewer, fall back to a safe default. The rest of the chapter is mostly about quantifying the second kind.


In [ ]:
# Zoom out from individual failures: how does confidence behave on the
# whole test set?
fig, ax = plt.subplots(figsize=(8, 4.5))
_evens = df.loc[df['parity'] == 'even', 'top1_softmax']
_odds  = df.loc[df['parity'] == 'odd',  'top1_softmax']
_bins  = np.linspace(0.2, 1.0, 33)
ax.hist(_evens, bins=_bins, color=_ACCENT, alpha=0.55,
        label=f'evens, in-distribution (n={len(_evens)})')
ax.hist(_odds,  bins=_bins, color=_TERRA,  alpha=0.55,
        label=f'odds,  out-of-distribution (n={len(_odds)})')
ax.set_xlabel('top-1 softmax probability')
ax.set_ylabel('count')
ax.set_title('Confidence on what the model knows vs what it does not',
             fontsize=10, color=_GOLDEN)
ax.legend(frameon=False, labelcolor=_TEXT)
tufte_axis(ax)
plt.tight_layout()
plt.show()


**Observe:**
- Both distributions concentrate at the right edge. The model is confident on evens (it should be) and almost as confident on odds (it should not be).
- The softmax was never built to detect novelty. It normalises across the classes the model knows; by construction it cannot say "none of the above".

Two definitions are useful before we start dismantling the trap.

**Anomaly detection.** Flag inputs that look unusual *within* the training distribution. A fraudulent transaction in a stream of normal ones, a defective part on an assembly line.

**Out-of-distribution (OOD) detection.** Flag inputs that are not from the training distribution at all. An odd digit handed to a classifier trained only on evens.

The two questions are different in general, but our setup deliberately collapses them. Odds are unusual *and* unrepresented. The same scoring tools therefore answer both questions on this dataset, and we will use that to keep the story tight. The next section starts with the cheapest of those tools.


## 🎲 MC Dropout

Dropout was a regulariser at training time: random subsets of activations were zeroed each batch so the network would not become brittle to any particular activation pathway. At test time it is usually turned off.

**Monte Carlo Dropout** turns it back on, deliberately, and runs `T` forward passes for each input. Each pass uses a different random mask and so is, in effect, a different sub-network drawn from a posterior over weights. The mean across the `T` passes is the predictive mean. The spread across them is the predictive variance, and it is what the deterministic classifier did not have.

> Look at one input the deterministic classifier put at p = 0.95 confidence. Across `T = 20` dropout-on passes, what would you expect the spread of probabilities to look like for an in-distribution input vs an OOD one?

<details><summary>Thought</summary>

For an in-distribution input the masks pinch off pathways the network does not depend on for that example; the prediction is robust and the `T` probabilities cluster tightly around the deterministic value. For an OOD input the network's prediction depends on accidental wiring; different masks flip the answer, and the `T` probabilities scatter across a wide range. The deterministic classifier averaged that scatter into a single confident-looking number; MC Dropout exposes it.
</details>

Implement `predictive_stats(samples)`. Given an array of shape `(T, n, K)` containing `T` forward-pass softmax tensors over `n` inputs and `K` classes, return a tuple `(mean, var)` each of shape `(n, K)`, averaging over the `T` axis.

Useful operations: `np.mean(..., axis=0)`, `np.var(..., axis=0)`.


In [ ]:
def predictive_stats(samples):
    """Aggregate T forward-pass predictions into a mean and a predictive variance.

    Parameters
    ----------
    samples : ndarray of shape (T, n, K)
        T forward-pass softmax tensors over n inputs and K classes.

    Returns
    -------
    (mean, var) : tuple of ndarrays, each of shape (n, K)
    """
    # YOUR CODE HERE
    pass


check_predictive_stats(predictive_stats)


In [ ]:
# Reshape the wide CSV into a (T, n, K) tensor for predictive_stats.
def _reshape_passes(df, prefix, n_passes, classes):
    """Stack {prefix}{p:02d}_c{c} columns into shape (n_passes, n, len(classes))."""
    out = np.empty((n_passes, len(df), len(classes)))
    for t in range(n_passes):
        for k, c in enumerate(classes):
            out[t, :, k] = df[f'{prefix}{t:02d}_c{c}'].values
    return out


def _auroc(scores, is_odd):
    """ROC-AUC for `scores` ranking odd-parity inputs above even ones."""
    order = np.argsort(scores)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(scores) + 1)
    n_pos = int(is_odd.sum()); n_neg = len(scores) - n_pos
    return float((ranks[is_odd].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


_T = 20
_classes = [0, 2, 4, 6, 8]
_mc = pd.read_csv('mc_dropout_p03.csv')
_passes = _reshape_passes(_mc, 't', _T, _classes)
print(f'mc dropout passes shape : {_passes.shape}   (T, n, K)')

_stats = predictive_stats(_passes)
if _stats is None:
    print('⬜ Implement predictive_stats above first.')
else:
    _mean, _var = _stats
    _var_total = _var.sum(axis=1)         # one OOD score per sample
    _is_odd    = (_mc['parity'] == 'odd').values
    _auc       = _auroc(_var_total, _is_odd)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    _bins  = np.linspace(0, _var_total.max(), 33)
    ax.hist(_var_total[~_is_odd], bins=_bins, color=_ACCENT, alpha=0.55,
            label=f'evens (n={(~_is_odd).sum()})')
    ax.hist(_var_total[ _is_odd], bins=_bins, color=_TERRA, alpha=0.55,
            label=f'odds  (n={ _is_odd.sum()})')
    ax.set_xlabel('predictive variance (sum across classes)')
    ax.set_ylabel('count')
    ax.set_title(f'MC Dropout variance, p = 0.3   (AUROC vs odds = {_auc:.3f})',
                 fontsize=10, color=_GOLDEN)
    ax.legend(frameon=False, labelcolor=_TEXT)
    tufte_axis(ax)
    plt.tight_layout()
    plt.show()


**Observe:**
- The two distributions separate where the softmax confidence histogram did not. Evens cluster near zero variance; odds spread to higher values.
- The signal is not perfect; the histograms overlap. Some odds happen to look like specific evens cleanly enough that the masks all agree, and they get a low variance score.
- The AUROC in the title is the rank-based summary: 1.0 means perfect separation, 0.5 means coin flip. The number is well above 0.5, which is what we wanted, but well below 1.0, which is what reality looks like.


### How much does the dropout rate matter?

The dropout rate `p` controls how much each pass differs from the next. At `p = 0` every pass is the same network and the variance is identically zero. At `p` close to `1` almost everything is zeroed each pass and the predictions are noise. Both extremes destroy the signal; the useful range is somewhere in between.

> Predict: as `p` grows, what should happen to the absolute variance for evens vs odds, and what should happen to the AUROC of variance-as-OOD-score?

<details><summary>Thought</summary>

The absolute variance grows with `p` for both groups. More perturbation, more disagreement, larger spread. The AUROC is rank-based, so it depends on the *ordering* of scores rather than their magnitudes; we should expect it to be flatter across rates than the histograms make it look. Picking a dropout rate is mostly about getting the absolute scale into a useful range, not about chasing a hidden sweet spot.
</details>


In [ ]:
# Side by side: variance histograms and the AUROC under each rate.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
_panels = []
_max_v = 0.0
for p_str, p_label in [('p01', '0.1'), ('p03', '0.3'), ('p05', '0.5')]:
    _df = pd.read_csv(f'mc_dropout_{p_str}.csv')
    _ps = _reshape_passes(_df, 't', _T, _classes)
    _, _v = predictive_stats(_ps)
    _vt = _v.sum(axis=1)
    _max_v = max(_max_v, float(_vt.max()))
    _panels.append((p_label, _df, _vt))

_bins = np.linspace(0, _max_v, 33)
for ax, (p_label, _df, _vt) in zip(axes, _panels):
    _is_odd = (_df['parity'] == 'odd').values
    _auc    = _auroc(_vt, _is_odd)
    ax.hist(_vt[~_is_odd], bins=_bins, color=_ACCENT, alpha=0.55, label='evens')
    ax.hist(_vt[ _is_odd], bins=_bins, color=_TERRA,  alpha=0.55, label='odds')
    ax.set_xlabel('predictive variance')
    ax.set_title(f'p = {p_label}   AUROC = {_auc:.3f}', fontsize=10, color=_GOLDEN)
    tufte_axis(ax)
axes[0].set_ylabel('count')
axes[-1].legend(frameon=False, labelcolor=_TEXT)
plt.tight_layout()
plt.show()


**Observe:**
- The absolute variance grows with `p`. At `p = 0.1` both distributions sit near the origin; at `p = 0.5` both are pushed several times further out.
- The AUROC numbers in the panel titles tell a flatter story. The rate matters far less than the histogram shapes make it look; what changes is the absolute scale, not the discriminative power.
- The lesson is rank-based: as long as the dropout rate is large enough that variance is non-trivial and small enough that in-distribution stays roughly consistent, the OOD-vs-in-distribution ordering is largely preserved.

Pick a rate that lands the absolute scale in a comfortable range for thresholding. Do not chase a hidden sweet spot in AUROC; on a setup like this one, there is none.


### A companion: test-time augmentation

MC Dropout perturbs the model. **Test-time augmentation (TTA)** perturbs the input. Run the same input through the same fixed network under several small, label-preserving augmentations (rotations of a few degrees, translations of a few pixels) and read off the disagreement.

The same `predictive_stats` you wrote above works on TTA passes without any change. The argument tensor still has shape `(T, n, K)`. Only the source of variation across the `T` axis differs: a different dropout mask, or a different augmented input.

> If MC Dropout asks "is my answer robust to which sub-network I happened to draw?", what does TTA ask, and which kinds of OOD shifts should each one catch best?

<details><summary>Thought</summary>

TTA asks "is my answer robust to a small change in the input?". It catches OOD shifts that look like nearby perturbations of training inputs (different lighting, slight rotation, mild occlusion) particularly well, because the augmentations are samples of those exact perturbations. It catches less well shifts that are not perturbations but different objects altogether — and that is exactly the situation our odds-vs-evens setup is in.
</details>


In [ ]:
_n_aug = 8
_tta = pd.read_csv('tta_passes.csv')
_tta_passes = _reshape_passes(_tta, 'a', _n_aug, _classes)
print(f'TTA passes shape : {_tta_passes.shape}   (T_aug, n, K)')

_, _v_tta = predictive_stats(_tta_passes)
_v_tta_total = _v_tta.sum(axis=1)
_is_odd_tta  = (_tta['parity'] == 'odd').values
_auc_tta     = _auroc(_v_tta_total, _is_odd_tta)

# Side by side with MC Dropout p = 0.3 for direct comparison.
_mc_p03 = pd.read_csv('mc_dropout_p03.csv')
_mc_passes = _reshape_passes(_mc_p03, 't', _T, _classes)
_, _v_mc = predictive_stats(_mc_passes)
_v_mc_total = _v_mc.sum(axis=1)
_is_odd_mc  = (_mc_p03['parity'] == 'odd').values
_auc_mc     = _auroc(_v_mc_total, _is_odd_mc)

# Softmax baseline from clf_predictions.csv: 1 - top1_softmax.
_pr  = pd.read_csv('clf_predictions.csv')
_sm  = (1.0 - _pr['top1_softmax']).values
_auc_sm = _auroc(_sm, (_pr['parity'] == 'odd').values)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, scores, is_odd, title, auc in [
    (axes[0], _v_mc_total,  _is_odd_mc,  'MC Dropout (p = 0.3)', _auc_mc),
    (axes[1], _v_tta_total, _is_odd_tta, 'TTA (8 augmentations)', _auc_tta),
]:
    _bins = np.linspace(0, float(scores.max()), 33)
    ax.hist(scores[~is_odd], bins=_bins, color=_ACCENT, alpha=0.55, label='evens')
    ax.hist(scores[ is_odd], bins=_bins, color=_TERRA,  alpha=0.55, label='odds')
    ax.set_xlabel('predictive variance')
    ax.set_title(f'{title}   AUROC = {auc:.3f}', fontsize=10, color=_GOLDEN)
    tufte_axis(ax)
axes[0].set_ylabel('count')
axes[-1].legend(frameon=False, labelcolor=_TEXT)
plt.tight_layout()
plt.show()
print(f'softmax baseline (1 - top1_softmax) AUROC = {_auc_sm:.3f}')


**Observe:**
- Both methods produce histograms that look like they should help. Both also separate evens from odds clearly enough to see by eye.
- The AUROC is the more honest readout. On this dataset MC Dropout is comparable to the softmax baseline, and TTA is meaningfully worse. The augmentations rotate and translate the digits, which is exactly the wrong knob for an OOD setup where evens and odds are different shapes, not slight perturbations of each other.
- The general lesson: predictive-variance methods catch the OOD shifts they are designed to perturb against. MC Dropout asks about parameter sensitivity; TTA asks about input-pose sensitivity. Pick the one whose perturbation matches the kind of shift you expect.

We have a usable but limited signal. The next section asks a different question: instead of perturbing the model or the input, can the model's own VAE-style reconstruction flag inputs it cannot represent?


## 🪞 Find the Odds (TODO)

## 🎁 The Misc Class (TODO)

## ⚡ Softmax vs Energy (TODO)

### 🏁 Recap (TODO)

## Take It from Here, Next Steps

### 🧭 LOF in Latent Space (TODO)

### 🚦 Decision Routing (TODO)

### 🛡️ Confidence Head (TODO)

### 🌡️ Calibration (TODO)